# x3e_infracoes — Termos de Embargo IBAMA (Histórico e Coordenadas)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** embargos (IBAMA), registros de histórico e coordenadas, localização pontual, datas dos registros*  
**Conceito e Mecanismo Econômico:** Medida cautelar que paralisa atividades produtivas em áreas desmatadas ilegalmente, gerando sanções comerciais e bloqueio de crédito rural.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\raw\tabular\ibama\termo_embargo_historico.csv`
- `2_INTI\xdata\chap1\raw\tabular\ibama\termo_de_embargo_coordenadas.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Candidato ainda parcial: vincular identidades, datas e municípios antes de construir contagem anual.
> - Linhas de histórico/coordenadas não são embargos únicos; área depende de unidade validada.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`SEQ_TAD`**: Identificador único do Termo de Embargo.
- **`DAT_EMBARGO`**: Data do embargo.
- **`QTD_AREA_EMBARGADA`**: Área interditada (ha).
- **`SIT_CANCELADO`**: Situação do cancelamento.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\raw\tabular\ibama\termo_embargo_historico.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=';', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x3e_infracoes`:
- **Atenção:** Candidato ainda parcial: vincular identidades, datas e municípios antes de construir contagem anual.
- **Atenção:** Linhas de histórico/coordenadas não são embargos únicos; área depende de unidade validada.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df['QTD_AREA_EMBARGADA'] = pd.to_numeric(df['QTD_AREA_EMBARGADA'].astype(str).str.replace(',', '.'), errors='coerce')
df['DAT_EMBARGO'] = pd.to_datetime(df['DAT_EMBARGO'], errors='coerce')
df['ano'] = df['DAT_EMBARGO'].dt.year

# Filtrar Mato Grosso e embargos não cancelados
df = df[(df['UF'] == 'MT') & (df['SIT_CANCELADO'] != 'S')].copy()
df['codigo_ibge'] = df['COD_MUNICIPIO'].astype(str).str.zfill(7).replace({'5101837': '5106240'})

# REGRA DO TXT: 'Linhas de histórico/coordenadas não são embargos únicos; área depende de unidade validada.'
df = df.drop_duplicates(subset=['SEQ_TAD'])
print(f"Total de embargos únicos validados em MT: {len(df)}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva da Área Embargada (ha):")
print(df['QTD_AREA_EMBARGADA'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
sns.histplot(df[df['QTD_AREA_EMBARGADA'] > 0]['QTD_AREA_EMBARGADA'], bins=30, log_scale=True, color='firebrick')
plt.title("Área Embargada por Termo (ha, Escala Log)")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
emb_ano = df.groupby('ano').size().reset_index(name='n_novos_embargos')
sns.lineplot(data=emb_ano[(emb_ano['ano'] >= 2005) & (emb_ano['ano'] <= 2024)], x='ano', y='n_novos_embargos', marker='o', color='darkred')
plt.title("Evolução de Novos Termos de Embargo em MT")
plt.show()

## Fase 6: Feature Engineering — Estoque Acumulado de Áreas Sob Embargo Ativo

### 1. Step 1: Fluxo vs Estoque Acumulado de Embargos
- **Diferença Teórica Fundamental:** Uma multa é paga ou contestada, mas o **embargo** perdura no tempo! Se 1.000 ha foram embargados em 2005, essa terra continua legalmente interditada para plantio e comércio em 2006, 2007 e 2020 (a menos que haja decisão de desembargo).
- Portanto, o controle correto não é apenas o fluxo de novos embargos no ano, mas o **estoque acumulado de áreas embargadas ativas**.

### 2. Step 2: Derivações
- **Estoque Acumulado Defasado:** $EstoqueEmbargo_{i, t-1} = \sum_{k=t_0}^{t-1} AreaEmbargada_{i, k}$.
- **Proporção Municipal Interditada:** $ShareEmbargado_{i, t-1} = \frac{EstoqueEmbargo_{i, t-1}}{AreaTotalMunicipio_i}$.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Agrupar novos embargos anuais por município
painel_embargo = df.groupby(['codigo_ibge', 'ano']).agg(
    n_novos_embargos=('SEQ_TAD', 'count'),
    area_embargada_ha=('QTD_AREA_EMBARGADA', 'sum')
).reset_index()

# 2. Grade balanceada 141 x 24 com zeros
mun_mt = df['codigo_ibge'].unique()
grade = pd.MultiIndex.from_product([mun_mt, range(2001, 2025)], names=['codigo_ibge', 'ano']).to_frame().reset_index(drop=True)
painel_embargo = pd.merge(grade, painel_embargo, on=['codigo_ibge', 'ano'], how='left').fillna(0)

painel_embargo.sort_values(['codigo_ibge', 'ano'], inplace=True)

# 3. Construção do Estoque Acumulado de Áreas Embargadas ao longo do tempo (cumsum)
# estoque_embargado_acum_ha: área total acumulada sob embargo até o ano t
painel_embargo['estoque_embargado_acum_ha'] = painel_embargo.groupby('codigo_ibge')['area_embargada_ha'].cumsum()

# 4. Defasagem Temporal de 1 ano (t-1)
painel_embargo['estoque_embargo_lag1'] = painel_embargo.groupby('codigo_ibge')['estoque_embargado_acum_ha'].shift(1).fillna(0)
painel_embargo['ln_estoque_embargo_lag1'] = np.log1p(painel_embargo['estoque_embargo_lag1'])

print("Estoque Acumulado de Áreas Interditadas por Embargo (t-1):")
print(painel_embargo[['codigo_ibge', 'ano', 'area_embargada_ha', 'estoque_embargo_lag1', 'ln_estoque_embargo_lag1']].head())